This notebook for finding best parmter on LSTM

This is to find best parm based on LSTM

#**Pre-request**

##Mount google drive


In [1]:
### **Mount** Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Install pakages


In [2]:
#Install pakages
project_path = "/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/"
!cat "{project_path}requirement/Install/NASEnhancedPretraindMLModleAdvance.txt"
!pip install  -r "{project_path}requirement/Install/NASEnhancedPretraindMLModleAdvance.txt" --no-cache-dir
%cd $project_path





torch
transformers
huggingface_hub
datasets
timm
patool
sktime
reformer_pytorch
optuna
ptflopsRequirement already satisfied: torch in /usr/local/lib/python3.12/dist-packages (from -r /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/requirement/Install/NASEnhancedPretraindMLModleAdvance.txt (line 1)) (2.11.0+cu128)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 343.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 386.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 372.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 350.4 MB/s eta 0:00:00
/content/drive/MyDrive/Sem-6/coding/github/fraud_detection


##Import  libs

In [3]:
# =====================================================
# 📦 Standard Library
# =====================================================
import os
import sys
import time
import logging
import hashlib
import shutil
import subprocess
import warnings
from datetime import datetime

# Start timer
start_time = time.time()

# =====================================================
# 🧮 Data & Visualization
# =====================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# =====================================================
# ⚙️ Machine Learning - Scikit-learn
# =====================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.utils import class_weight
from sklearn.covariance import EmpiricalCovariance
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

# =====================================================
# 🌲 XGBoost
# =====================================================
from xgboost import XGBClassifier
import joblib

# =====================================================
# 🔥 Deep Learning - PyTorch
# =====================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.cuda.amp import autocast

# =====================================================
# 🤖 Deep Learning - TensorFlow / Keras
# =====================================================
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Masking, Dropout, Layer
from tensorflow.keras.optimizers import Adam

# =====================================================
# 🤗 Transformers & Time Series
# =====================================================
from transformers import AutoModel
from sktime.datasets import load_from_tsfile_to_dataframe
# from mamba_ssm import Mamba  # Uncomment if needed

# =====================================================
# 🧠 Explainability
# =====================================================
import shap

# =====================================================
# 📊 Google Colab Specific
# =====================================================
from google.colab import data_table
data_table.enable_dataframe_formatter()
try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
    data_table.DataTable.max_columns = 50
    data_table.DataTable.max_rows = 150
except ImportError:
    pass
from tqdm import tqdm

print("✅ All imports loaded successfully!")

✅ All imports loaded successfully!


##Confirmation setup

In [4]:
# =====================================================
# 🎲 Random Seeds (Reproducibility)
# =====================================================
!nvidia-smi                # confirm GPU
!pip show torch  # confirm versions
torch.manual_seed(42)
np.random.seed(42)

Thu Jul  9 11:31:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   27C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Enable Config

In [5]:

logger = logging.getLogger(__name__)

def load_config(config_path="configs/baseline.yaml"):
    """Load YAML config file and expand ${root_path} placeholders."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    logger.info(f"✅ Loaded config from {config_path}")

    # --- Expand ${root_path} placeholders ---
    root = config.get("root_path", "")

    def expand_paths(obj):
        if isinstance(obj, dict):
            return {k: expand_paths(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [expand_paths(i) for i in obj]
        elif isinstance(obj, str) and "${root_path}" in obj:
            return obj.replace("${root_path}", root)
        else:
            return obj

    config = expand_paths(config)
    return config
config = load_config(os.path.join(project_path, "configs", "baseline.yaml"))


## Set Variables

In [6]:


#limit = config['ML']['limit']

# ==========================================================
# UNIFIED HYPERPARAMETERS (Match TimesNet NAS Best)
# ==========================================================
# ==========================================================
# 🔧 UNIFIED HYPERPARAMETERS (Match TimesNet NAS Best)
# ==========================================================

# ----------------------------------------------------------
# 📏 Sequence Settings
# ----------------------------------------------------------
max_seq_len = 6                  # Maximum sequence length
recent_mode = False               # False → oldest mode, True → recent-window mode
nas_epochs = 20
n_trials_rf_xgb = 100




# ----------------------------------------------------------
# 📊 Evaluation Settings
# ----------------------------------------------------------
threshold = 0.5                   # Default classification threshold
opt_metric = "f1"                 # Optimization metric for model selection
early_stop_metric = 'val_accuracy'# Metric for early stopping
correlation_threshold = 0.85      # Feature correlation threshold


# ----------------------------------------------------------
# 💾 Paths
# ----------------------------------------------------------
model_path = config['ML']['models']

# ----------------------------------------------------------
# 🔄 Training State (Reset before each model)
# ----------------------------------------------------------
best_threshold = 0.5              # Best threshold (general)

# ==========================================================
# ✅ Configuration Summary
# ==========================================================
print("="*60)
print("📋 CONFIGURATION SUMMARY")
print("="*60)
print(f"  Sequence length:  {max_seq_len}")
print(f"  Mode:             {'Recent' if recent_mode else 'Oldest'}")
print(f"  Threshold:        {threshold}")
print(f"  Model path:       {model_path}")
print("="*60)

# Global unified results table for all models
results_table = pd.DataFrame(columns=["Round", "AUC", "Recall", "F1", "Model"])
summary = pd.DataFrame(
    columns=[
        "Model",
        "AUC",
        "Recall",
        "Precision",
        "F1",
        "threshold"
    ]
)


📋 CONFIGURATION SUMMARY
  Sequence length:  6
  Mode:             Oldest
  Threshold:        0.5
  Model path:       /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/models/


##Split users level

In [7]:

# user_path = config['ML']['Events']['base_path'] + config['ML']['Events']['files']['user']
# df_user = pd.read_csv(user_path)
# print(f"✅ Loaded transactional user dataset: {df_user.shape}")



# # Aggregate to one row per user (max label = 1 if any fraud)
# user_labels = df_user.groupby("phone_no_m")["label"].max()
# print(f"👥 Unique users for splitting: {len(user_labels)}")

# # ==============================================================
# # 2️⃣ Create user-level split (stratified, no leakage)
# # ==============================================================

# fraud_users = user_labels[user_labels == 1].index
# normal_users = user_labels[user_labels == 0].index

# fraud_train, fraud_test = train_test_split(fraud_users, test_size=0.2, random_state=42)
# normal_train, normal_test = train_test_split(normal_users, test_size=0.2, random_state=42)

# train_users = set(fraud_train) | set(normal_train)
# test_users  = set(fraud_test)  | set(normal_test)

# # ==============================================================
# # 3️⃣ Save unified split (shared across LSTM / RF / XGB)
# # ==============================================================

# split_dir = "splits/shared_user_split_v1"
# os.makedirs(split_dir, exist_ok=True)

# pd.DataFrame({"phone_no_m": sorted(train_users)}).to_csv(f"{split_dir}/train_users.csv", index=False)
# pd.DataFrame({"phone_no_m": sorted(test_users)}).to_csv(f"{split_dir}/test_users.csv", index=False)

# # ==============================================================
# # 4️⃣ Summary
# # ==============================================================

# print("\n👥 Users Summary:")
# print(f"   Total : {len(user_labels):,}")
# print(f"   Fraud : {len(fraud_users):,} ({len(fraud_users)/len(user_labels)*100:.2f}%)")
# print(f"   Normal: {len(normal_users):,} ({len(normal_users)/len(user_labels)*100:.2f}%)")

# print("\n📂 Split saved to /splits/:")
# print(f"   Train users: {len(train_users)}")
# print(f"   Test  users: {len(test_users)}")
# print(f"   Fraud ratio train: {len(fraud_train)/len(train_users)*100:.2f}%")
# print(f"   Fraud ratio test : {len(fraud_test)/len(test_users)*100:.2f}%")


## Helpers

### evaluate_global

In [8]:
def evaluate_global(model, X_test, y_test, model_name="Model", threshold=0.5):
    """
    Generic evaluator for both classic ML models and neural networks.
    """
    print(f"\n📊 Evaluation threshold is: {threshold}")

    # ---- Predict probabilities ----
    if hasattr(model, "predict_proba"):
        # For sklearn-style models
        y_pred_prob = model.predict_proba(X_test)[:, 1]
    else:
        # For neural nets (e.g., Keras)
        preds = model.predict(X_test)
        if preds.shape[-1] == 2:
            # 2-class softmax output
            y_pred_prob = preds[:, 1]
        else:
            # Single sigmoid output
            y_pred_prob = preds.ravel()

    # ... rest of function unchanged
    # ---- Predict classes ----
    y_pred = (y_pred_prob > threshold).astype(int)

    # ---- Metrics ----
    auc = roc_auc_score(y_test, y_pred_prob)
    #recall = recall_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, zero_division=0)

    precision = precision_score(y_test, y_pred, zero_division=0)
    #f1 = f1_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, zero_division=0)

    report = classification_report(y_test, y_pred, digits=4)
    cm = confusion_matrix(y_test, y_pred)

    # ---- Display ----
    print(f"\n📊 Classification Report — {model_name}")
    print(report)
    print(f"AUC: {auc:.4f} | Recall: {recall:.4f} | Precision: {precision:.4f} | F1: {f1:.4f}")

    # ---- Confusion Matrix ----
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal (0)", "Fraud (1)"])
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix — {model_name}")
    plt.grid(False)
    plt.show()

    # ---- Summary Dictionary ----
    return {
        "Model": model_name,
        "AUC": auc,
        "Recall": recall,
        "Precision": precision,
        "F1": f1,
        "threshold": threshold
    }



### append_to_summary

In [9]:

def append_to_summary(summary, model_name, results):
    """
    Appends or updates the summary table with model results.
    Works with both capitalized and lowercase keys automatically.
    """
    # ✅ Create summary DataFrame if missing
    if summary is None or not isinstance(summary, pd.DataFrame):
          summary = pd.DataFrame(columns=["Model", "AUC", "Recall", "Precision", "F1", "Threshold"])
    # Ensure "Model" column exists (prevents KeyError)
    if "Model" not in summary.columns:
        summary = pd.DataFrame(columns=["Model", "AUC", "Recall", "Precision", "F1", "Threshold"])

    # ✅ Normalize key names to lowercase
    results = {k.lower(): v for k, v in results.items()}

    # ✅ Remove any existing row for the same model
    summary = summary[summary["Model"] != model_name]

    # ✅ Add new row (robust to missing values)
    row = {
        "Model": model_name,
        "AUC": round(results.get("auc", np.nan), 4) if not pd.isna(results.get("auc", np.nan)) else np.nan,
        "Recall": round(results.get("recall", np.nan), 4) if not pd.isna(results.get("recall", np.nan)) else np.nan,
        "Precision": round(results.get("precision", np.nan), 4) if not pd.isna(results.get("precision", np.nan)) else np.nan,
        "F1": round(results.get("f1", np.nan), 4) if not pd.isna(results.get("f1", np.nan)) else np.nan,
        "Threshold": round(results.get("threshold", np.nan), 4) if not pd.isna(results.get("threshold", np.nan)) else np.nan
    }


    # ✅ Append and reindex
    summary = pd.concat([summary, pd.DataFrame([row])], ignore_index=True)
    summary = summary.reindex(columns=["Model", "AUC", "Recall", "Precision", "F1", "Threshold"])
    return summary


### find_best_threshold

In [10]:
def find_best_threshold(y_true, probs, low=0.2, high=0.8, n=61):
    best_f1 = -1.0
    best_thr = 0.5
    for thr in np.linspace(low, high, n):
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
    return best_thr, best_f1


def get_timesnet_val_threshold(results_dir):
    val_prob_path = os.path.join(results_dir, "val_prob.npy")
    val_true_path = os.path.join(results_dir, "val_true.npy")

    if not os.path.exists(val_prob_path) or not os.path.exists(val_true_path):
        raise FileNotFoundError(
            f"Missing validation probability files in {results_dir}. "
            f"Expected val_prob.npy and val_true.npy"
        )

    val_probs = np.load(val_prob_path)
    val_true = np.load(val_true_path)

    best_thr, best_f1 = find_best_threshold(val_true, val_probs)
    return best_thr, best_f1

###Drop and select features

In [11]:
def prepare_features(df):
    """
    Selects only the explicitly defined features for model training.
    You control which features are used by editing 'selected_features' below.
    """

    # --- Define selected features manually ---
    selected_features = [
        "window_size", "voc_total_calls", "voc_unique_contacts", "voc_total_duration",
       "voc_avg_duration", "voc_max_duration", "voc_std_duration", "voc_active_days",
       "voc_active_hours", "sms_total_msgs", "sms_unique_contacts", "sms_active_hours",
       "sms_calltype_ratio", "app_months_active", "app_total_flow", "app_avg_flow",
       "app_std_flow", "app_unique_apps_mean", "app_unique_apps_max", "user_months_active",
        "arpu_mean", "arpu_std", "arpu_max", "idcard_cnt", "snapshot_round"
   ]
  #  selected_features = [
   #     "voc_total_calls", "voc_unique_contacts", "voc_total_duration",
    #   "voc_avg_duration", "voc_max_duration", "voc_std_duration", "voc_active_days",
     # "voc_active_hours", "sms_total_msgs", "sms_unique_contacts", "sms_active_hours",
     #"sms_calltype_ratio", "idcard_cnt"
    #]
   # selected_features = [
    #    "voc_active_days",
    #"voc_active_hours",
    #"voc_unique_contacts",
    #"sms_calltype_ratio",
    #"sms_active_hours" ]


    # ✅ You can manually remove or comment out features here
    # For example:
    # selected_features = [f for f in selected_features if not (f.startswith("app_") or f.startswith("arpu_"))]

    # --- Keep only existing columns ---
    available = [f for f in selected_features if f in df.columns]
    missing = [f for f in selected_features if f not in df.columns]

    X = df[available].copy()

    #print(f"\n📊 Final features used ({len(available)}): {available}")
    if missing:
        print(f"⚠️ Missing columns not found in data: {missing}")

    return X


### Compare

In [12]:

def plot_progressive_results(
    results_table,
    metrics=("AUC", "Recall", "F1"),
    round_col=None,
    figsize=(14, 6)
):
    """
    Plot progressive evaluation metrics per round for multiple models.

    Parameters
    ----------
    results_table : pd.DataFrame
        Must contain columns: Model, metrics, and either Round or input_size
    metrics : tuple
        Metrics to plot (default: AUC, Recall, F1)
    round_col : str or None
        Column name for x-axis. If None, auto-detects.
    figsize : tuple
        Figure size for plots
    """

    # --------------------------------------------------
    # Auto-detect round column
    # --------------------------------------------------
    if round_col is None:
        if "Round" in results_table.columns:
            round_col = "Round"
        elif "input_size" in results_table.columns:
            round_col = "input_size"
        else:
            raise ValueError("No Round or input_size column found.")

    # --------------------------------------------------
    # Sort results (important for correct curves)
    # --------------------------------------------------
    results_table = results_table.sort_values(
        by=[round_col, "Model"],
        ascending=True
    ).reset_index(drop=True)


    # --------------------------------------------------
    # Plot each metric
    # --------------------------------------------------
    for metric in metrics:

        plt.figure(figsize=figsize)

        for model in results_table["Model"].unique():
            subset = (
                results_table[results_table["Model"] == model]
                .sort_values(by=round_col)
            )

            plt.plot(
                subset[round_col],
                subset[metric],
                marker="o",
                markersize=6,
                linewidth=2,
                label=model,
                alpha=0.85
            )

        plt.title(f"{metric} per {round_col}", fontsize=18)
        plt.xlabel(round_col, fontsize=14)
        plt.ylabel(metric, fontsize=14)
        plt.grid(True, linestyle="--", alpha=0.4)

        # Legend outside
        plt.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.12),
            ncol=4,
            fontsize=10
        )

        plt.tight_layout(rect=[0, 0.1, 1, 1])
        plt.show()
    display(results_table)

    return results_table


###get_key_rounds

In [13]:
# ============================================================
# 🔬 SCIENTIFIC KEY ROUNDS SELECTION
# ============================================================

def get_key_rounds(max_seq_len, method="linear", n_points=5):
    """
    Generate scientifically meaningful evaluation checkpoints.

    Args:
        max_seq_len: Maximum sequence length (16, 100, 300, etc.)
        method: Selection strategy
            - "linear": Equal spacing (1, 25%, 50%, 75%, 100%)
            - "logarithmic": More points early (where changes happen fast)
            - "sqrt": Square root spacing (balanced)
            - "percentile": Fixed percentages
        n_points: Number of evaluation points (default 5)

    Returns:
        List of round numbers to evaluate
    """

    if max_seq_len <= n_points:
        # If sequence is short, evaluate all rounds
        return list(range(1, max_seq_len + 1))

    if method == "linear":
        # Equal spacing: 1, 25%, 50%, 75%, 100%
        rounds = np.linspace(1, max_seq_len, n_points)

    elif method == "logarithmic":
        # More points early (fraud detection often shows early signal)
        # Log scale: 1, 2, 4, 8, 16 style
        rounds = np.logspace(0, np.log10(max_seq_len), n_points)

    elif method == "sqrt":
        # Square root spacing (balanced between linear and log)
        rounds = np.square(np.linspace(1, np.sqrt(max_seq_len), n_points))

    elif method == "percentile":
        # Fixed percentages: 1st event, 10%, 25%, 50%, 75%, 100%
        percentages = [0, 0.1, 0.25, 0.5, 0.75, 1.0]
        rounds = [max(1, int(p * max_seq_len)) for p in percentages]
        return sorted(set(rounds))  # Remove duplicates

    elif method == "early_focus":
        # Focus on early detection (more points in first half)
        # Useful for fraud detection where early signal matters
        early = np.linspace(1, max_seq_len * 0.5, n_points - 2)
        late = [max_seq_len * 0.75, max_seq_len]
        rounds = np.concatenate([early, late])

    else:
        raise ValueError(f"Unknown method: {method}")

    # Convert to integers, ensure valid range, remove duplicates
    rounds = [int(round(r)) for r in rounds]
    rounds = [max(1, min(r, max_seq_len)) for r in rounds]
    rounds = sorted(set(rounds))

    # Always include 1 and max_seq_len
    if 1 not in rounds:
        rounds = [1] + rounds
    if max_seq_len not in rounds:
        rounds = rounds + [max_seq_len]

    return rounds

##key_rounds = get_key_rounds(max_seq_len, method=method, n_points=n_points)
##print(f"📊 Evaluating rounds: {key_rounds}")
#print(f"   Total: {len(key_rounds)} rounds instead of {max_seq_len}")

#ML Modules

### Feature Importance

In [14]:
# def plot_feature_importance(model, X_train, model_name="Model", top_n=20):
#     """
#     Plot feature importance for tree-based models (XGBoost, RandomForest).
#     """


#     # Handle model type
#     if hasattr(model, "get_booster"):  # XGBoost
#         importance = model.get_booster().get_score(importance_type='gain')
#         fi = pd.DataFrame({
#             'Feature': list(importance.keys()),
#             'Importance': list(importance.values())
#         })
#     elif hasattr(model, "feature_importances_"):  # RandomForest
#         fi = pd.DataFrame({
#             'Feature': X_train.columns,
#             'Importance': model.feature_importances_
#         })
#     else:
#         raise ValueError(f"{model_name} does not support feature importance extraction.")

#     # Sort and plot
#     fi = fi.sort_values(by='Importance', ascending=False)
#     display(fi.head(10))

#     plt.figure(figsize=(10,6))
#     plt.barh(fi['Feature'][:top_n][::-1], fi['Importance'][:top_n][::-1])
#     plt.title(f'📊 {model_name} Feature Importance (Top {top_n})')
#     plt.xlabel('Importance')
#     plt.ylabel('Feature')
#     plt.grid(alpha=0.4)
#     plt.tight_layout()
#     plt.show()

#     return fi

# fi_xgb = plot_feature_importance(xgb_model, snap_X_train, "XGBoost")
# fi_rf = plot_feature_importance(rf_model, snap_X_train, "Random Forest")


### Load

In [15]:
def load_raw_datasets(config):


    if "ML" in config and "Events" in config["ML"]:
        events_cfg = config["ML"]["Events"]
    else:
        events_cfg = config["Events"]

    base = events_cfg["base_path"]
    files = events_cfg["files"]

    # --- Load all datasets ---
    df_voc = pd.read_csv(os.path.join(base, files["voc"]))
    df_sms = pd.read_csv(os.path.join(base, files["sms"]))
    df_app = pd.read_csv(os.path.join(base, files["app"]))
    df_user = pd.read_csv(os.path.join(base, files["user"]))

    # --- Normalize timestamps and add source column ---
    for df, src in [(df_voc, "VOC"), (df_sms, "SMS"), (df_app, "APP"), (df_user, "USER")]:
        df["source"] = src
        ts_col = [c for c in df.columns if "time" in c.lower()][0]
        df.rename(columns={ts_col: "event_time"}, inplace=True)
        df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")

    print("✅ Raw datasets loaded and timestamp-normalized.")
    return df_voc, df_sms, df_app, df_user

df_voc, df_sms, df_app, df_user = load_raw_datasets(config)


✅ Raw datasets loaded and timestamp-normalized.


### Build timeline (events)

In [16]:
def merge_and_prepare_events(df_voc, df_sms, df_app, df_user):

    # --- 1️⃣ Normalize USER dataset ---
    if 'label' not in df_user.columns:
        raise KeyError("❌ 'label' column not found in user dataset")

    # Ensure numeric consistency
    df_user['label'] = df_user['label'].fillna(0).astype(int)
    df_user['idcard_cnt'] = df_user['idcard_cnt'].fillna(0).astype(float)
    df_user['arpu_value'] = df_user['arpu_value'].fillna(0).astype(float)

    # --- 2️⃣ Extract static info for merging (label + sim count only) ---
    #static_user_info = df_user.groupby("phone_no_m", as_index=False)[["label", "idcard_cnt"]].max()
    # --- 2️⃣ Extract static info from the RAW user table (covers all 6,106 users) ---
    lbl_cfg = config["ML"]["labels"]
    raw_user = pd.read_csv(os.path.join(lbl_cfg["base_path"], lbl_cfg["file"]))
    static_user_info = raw_user[["phone_no_m", "label", "idcard_cnt"]].drop_duplicates("phone_no_m")
    static_user_info["label"] = static_user_info["label"].astype(int)
    static_user_info["idcard_cnt"] = static_user_info["idcard_cnt"].fillna(0).astype(float)

    # --- 3️⃣ Merge static info into other event tables ---
    df_voc = df_voc.merge(static_user_info, on="phone_no_m", how="left")
    df_sms = df_sms.merge(static_user_info, on="phone_no_m", how="left")
    df_app = df_app.merge(static_user_info, on="phone_no_m", how="left")


    # --- 4️⃣ Combine all transactional event sources ---
    # include df_user itself since arpu_value is event-like
    events = pd.concat([df_voc, df_sms, df_app, df_user], ignore_index=True)

    # --- 5️⃣ Fill and order ---
    #events["label"] = events["label"].fillna(0).astype(int)
    missing = int(events["label"].isna().sum())
    assert missing == 0, f"{missing} events have no label — label merge is broken"
    events["label"] = events["label"].astype(int)

    events["event_time"] = pd.to_datetime(events["event_time"], errors="coerce")
    events = events.sort_values(["phone_no_m", "event_time"]).reset_index(drop=True)

    # --- 6️⃣ Summary ---
    print("\n🔎 Feature Summary per Source:")
    for src, df in [("VOC", df_voc), ("SMS", df_sms), ("APP", df_app), ("USER", df_user)]:
        print(f"\n📂 Source: {src}")
        print(f"   Events: {len(df):,}")
        print(f"   Users : {df['phone_no_m'].nunique():,}")
        print(f"   Columns ({len(df.columns)}): {', '.join(df.columns)}")

    print("\n📊 Combined Dataset Summary:")
    print(f"   Total events: {len(events):,}")
    print(f"   Unique users: {events['phone_no_m'].nunique():,}")
    print(f"   Fraud ratio: {events['label'].mean()*100:.2f}%")
    user_labels = events.groupby("phone_no_m")["label"].max()
    print(f"   Fraud users: {int(user_labels.sum()):,} / {user_labels.size:,} ({user_labels.mean()*100:.2f}%)")

    return events

events = merge_and_prepare_events(df_voc, df_sms, df_app, df_user)

display(events.head())


🔎 Feature Summary per Source:

📂 Source: VOC
   Events: 5,015,430
   Users : 6,025
   Columns (11): phone_no_m, opposite_no_m, calltype_id, event_time, call_dur, city_name, county_name, imei_m, source, label, idcard_cnt

📂 Source: SMS
   Events: 6,848,509
   Users : 6,103
   Columns (7): phone_no_m, opposite_no_m, calltype_id, event_time, source, label, idcard_cnt

📂 Source: APP
   Events: 3,283,602
   Users : 6,106
   Columns (10): phone_no_m, event_time, source, busi_name, flow, month_id, flow_norm, month_str, label, idcard_cnt

📂 Source: USER
   Events: 39,454
   Users : 5,929
   Columns (10): phone_no_m, event_time, source, month_id, arpu_value, city_name, county_name, idcard_cnt, label, month_col

📊 Combined Dataset Summary:
   Total events: 15,186,995
   Unique users: 6,106
   Fraud ratio: 24.63%
   Fraud users: 1,962 / 6,106 (32.13%)


,phone_no_m,opposite_no_m,calltype_id,event_time,call_dur,city_name,county_name,imei_m,source,label,idcard_cnt,busi_name,flow,month_id,flow_norm,month_str,arpu_value,month_col
0,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-12 08:09:11,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-12 08:09:11,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-12 08:09:11,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-13 16:21:53,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-13 16:21:53,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Split data based on users (fraud, not fraud)

In [17]:


# ======================================
# Clean Numeric Columns
# ======================================
events = events.copy()
numeric_cols = events.select_dtypes(include=["number"]).columns.difference(["label"])

# Replace NaN with 0 for numeric fields (avoids scaling issues)
events[numeric_cols] = events[numeric_cols].fillna(0)

print(f"\n📊 Numeric columns to scale ({len(numeric_cols)}): {numeric_cols.tolist()}")


# ======================================
# 2 Create / Load Shared Train-Val-Test User Split
# ======================================
split_dir = os.path.join(config["root_path"], "splits", "shared_user_split_v1")
train_split_file = f"{split_dir}/train_users.csv"
test_split_file  = f"{split_dir}/test_users.csv"
val_split_file   = f"{split_dir}/val_users.csv"

os.makedirs(split_dir, exist_ok=True)

all_split_files_exist = (
    os.path.exists(train_split_file)
    and os.path.exists(test_split_file)
    and os.path.exists(val_split_file)
)

if all_split_files_exist:
    print("📂 Using shared user split files...")
    train_users = set(pd.read_csv(train_split_file)["phone_no_m"])
    test_users  = set(pd.read_csv(test_split_file)["phone_no_m"])
    val_users   = set(pd.read_csv(val_split_file)["phone_no_m"])
else:
    print("🆕 Creating shared train/test/val user split...")

    # One label per user
    user_labels = events.groupby("phone_no_m")["label"].max()
    fraud_users = user_labels[user_labels == 1].index
    normal_users = user_labels[user_labels == 0].index

    # 1) Train/Test split (split fraud user %)
    fraud_train, fraud_test = train_test_split(
        fraud_users, test_size=0.2, random_state=42
    )
    normal_train, normal_test = train_test_split(
        normal_users, test_size=0.2, random_state=42
    )

    train_users_full = set(fraud_train) | set(normal_train)
    test_users = set(fraud_test) | set(normal_test)

    # 2) Validation split from training users only
    train_user_labels = user_labels.loc[list(train_users_full)]

    fraud_train_users = train_user_labels[train_user_labels == 1].index
    normal_train_users = train_user_labels[train_user_labels == 0].index

    fraud_tr, fraud_val = train_test_split(
        fraud_train_users, test_size=0.2, random_state=42
    )
    normal_tr, normal_val = train_test_split(
        normal_train_users, test_size=0.2, random_state=42
    )

    train_users = set(fraud_tr) | set(normal_tr)
    val_users   = set(fraud_val) | set(normal_val)

    pd.DataFrame({"phone_no_m": sorted(train_users)}).to_csv(train_split_file, index=False)
    pd.DataFrame({"phone_no_m": sorted(test_users)}).to_csv(test_split_file, index=False)
    pd.DataFrame({"phone_no_m": sorted(val_users)}).to_csv(val_split_file, index=False)

    print(f"✅ Saved shared split files to '{split_dir}/'")
# ======================================
# Split overlap checks
# ======================================
assert len(train_users & test_users) == 0, "❌ User leakage: train and test overlap"
assert len(train_users & val_users) == 0, "❌ User leakage: train and val overlap"
assert len(test_users & val_users) == 0, "❌ User leakage: test and val overlap"

print("🔒 Split overlap checks passed:")
print(f"   train ∩ test = {len(train_users & test_users)}")
print(f"   train ∩ val  = {len(train_users & val_users)}")
print(f"   test  ∩ val  = {len(test_users & val_users)}")
user_labels = events.groupby("phone_no_m")["label"].max()
print(f"   sizes  train/val/test = {len(train_users)} / {len(val_users)} / {len(test_users)}")
print(f"   fraud  train/val/test = {int(user_labels.loc[list(train_users)].sum())} / "
      f"{int(user_labels.loc[list(val_users)].sum())} / {int(user_labels.loc[list(test_users)].sum())}")
# ======================================
# Apply Split to Events
# ======================================
train_events = events[events["phone_no_m"].isin(train_users)]
test_events  = events[events["phone_no_m"].isin(test_users)]
val_events = events[events["phone_no_m"].isin(val_users)]

# Sanity checks
assert len(set(train_events["phone_no_m"]) & set(test_events["phone_no_m"])) == 0, "❌ User leakage detected!"
assert train_events["label"].nunique() == 2, "❌ Training set must contain both classes"
assert test_events["label"].nunique() == 2, "❌ Test set must contain both classes"

# --- add time gap, scaled featur ---
# for name, df in [('train_events', train_events), ('test_events', test_events)]:
#     df = df.copy()  # avoid SettingWithCopyWarning
#     df['event_time'] = pd.to_datetime(df['event_time'])
#     #df.sort_values(['phone_no_m', 'event_time'], inplace=True)
#     df = df.sort_values(['phone_no_m', 'event_time'], kind='mergesort').reset_index(drop=True)
#     df['dt_hours'] = df.groupby('phone_no_m')['event_time'].diff().dt.total_seconds() / 3600
#     df['dt_hours'] = df['dt_hours'].fillna(0)
#     df['dt_hours'] = np.log1p(df['dt_hours'])  # normalize gaps
#     if name == 'train_events':
#         train_events = df
#     else:
#         test_events = df
for name, df in [
    ('train_events', train_events),
    ('val_events', val_events),
    ('test_events', test_events)
]:
    df = df.copy()
    df['event_time'] = pd.to_datetime(df['event_time'])
    df = df.sort_values(['phone_no_m', 'event_time'], kind='mergesort').reset_index(drop=True)
    df['dt_hours'] = df.groupby('phone_no_m')['event_time'].diff().dt.total_seconds() / 3600
    df['dt_hours'] = df['dt_hours'].fillna(0)
    df['dt_hours'] = np.log1p(df['dt_hours'])

    if name == 'train_events':
        train_events = df
    elif name == 'val_events':
        val_events = df
    else:
        test_events = df


# Store unscaled events BEFORE line 895
train_events_unscaled = train_events.copy()
test_events_unscaled = test_events.copy()
val_events_unscaled = val_events.copy()


# Sanity checks
assert len(set(train_events["phone_no_m"]) & set(test_events["phone_no_m"])) == 0, "❌ User leakage detected!"
assert train_events["label"].nunique() == 2, "❌ Training set must contain both classes"
assert test_events["label"].nunique() == 2, "❌ Test set must contain both classes"
scale_cols_seq = sorted(set(train_events.select_dtypes(include=["number"]).columns) - {"label"})
scaler_seq = StandardScaler().fit(train_events[scale_cols_seq])
train_events[scale_cols_seq] = scaler_seq.transform(train_events[scale_cols_seq])
test_events[scale_cols_seq]  = scaler_seq.transform(test_events[scale_cols_seq])
val_events[scale_cols_seq]   = scaler_seq.transform(val_events[scale_cols_seq])

# ======================================
# 4️⃣ snapshot
# ======================================

# ======================================
# 4️⃣ Create Sequences (using multiple features)
# ======================================
numeric_features = [c for c in numeric_cols if c not in ["label"]]  # exclude label
if 'dt_hours' in train_events.columns:
    numeric_features.append('dt_hours')
print(f"\n📦 Features used for sequences: {numeric_features}")
feature_cols_tf = numeric_features.copy()
# 👉 Transformer feature columns: same numeric features as LSTM + source_id

if "source_id" not in feature_cols_tf:
    feature_cols_tf.append("source_id")

print("[Transformer] feature_cols_tf:", feature_cols_tf)



📊 Numeric columns to scale (6): ['arpu_value', 'call_dur', 'calltype_id', 'flow', 'flow_norm', 'idcard_cnt']
📂 Using shared user split files...
🔒 Split overlap checks passed:
   train ∩ test = 0
   train ∩ val  = 0
   test  ∩ val  = 0
   sizes  train/val/test = 3907 / 977 / 1222
   fraud  train/val/test = 1255 / 314 / 393

📦 Features used for sequences: ['arpu_value', 'call_dur', 'calltype_id', 'flow', 'flow_norm', 'idcard_cnt', 'dt_hours']
[Transformer] feature_cols_tf: ['arpu_value', 'call_dur', 'calltype_id', 'flow', 'flow_norm', 'idcard_cnt', 'dt_hours', 'source_id']


### Generate_snapshots_from_events

In [18]:
# ============================================================
# 🔧 OPTIMIZED SNAPSHOT GENERATION FROM EVENTS (FIXED)
# ============================================================

def generate_snapshots_from_events(
    events_df,
    users,
    r,
    max_seq_len,
    recent_mode=True
):
    """
    Generate snapshot features from events DataFrame.
    OPTIMIZED: Uses groupby instead of per-user filtering.
    """

    # 1️⃣ Filter to relevant users FIRST (huge speedup)
    events_filtered = events_df[events_df["phone_no_m"].isin(users)].copy()

    if events_filtered.empty:
        return pd.DataFrame(), np.array([]), np.array([])

    # 2️⃣ Sort once
    events_filtered = events_filtered.sort_values(["phone_no_m", "event_time"])

    # 3️⃣ Apply selection logic per user using groupby
    def select_events(df_u):
        if recent_mode:
            df_u = df_u.tail(max_seq_len).head(r)
        else:
            df_u = df_u.head(r)
        return df_u

    selected = events_filtered.groupby("phone_no_m", group_keys=False).apply(select_events)

    if selected.empty:
        return pd.DataFrame(), np.array([]), np.array([])

    # 4️⃣ Aggregate features using groupby
    snapshot_rows = []

    for user, df_u in tqdm(selected.groupby("phone_no_m"), desc="Generating snapshots"):
    #for user, df_u in selected.groupby("phone_no_m"):
        row = {"phone_no_m": user}

        # Get label
        label = int(df_u["label"].max()) if "label" in df_u.columns else 0

        # --- Voice Features ---
        voc = df_u[df_u["source"] == "VOC"]
        if len(voc) > 0:
            if "call_dur" in voc.columns:
                call_dur = pd.to_numeric(voc["call_dur"], errors="coerce").fillna(0)
            else:
                call_dur = pd.Series([0])

            row["voc_total_calls"] = len(voc)
            row["voc_unique_contacts"] = voc["opposite_no_m"].nunique() if "opposite_no_m" in voc.columns else 0
            row["voc_total_duration"] = call_dur.sum()
            row["voc_avg_duration"] = call_dur.mean()
            row["voc_max_duration"] = call_dur.max()
            row["voc_std_duration"] = call_dur.std() if len(call_dur) > 1 else 0
            row["voc_active_days"] = voc["event_time"].dt.weekday.nunique()
            row["voc_active_hours"] = voc["event_time"].dt.hour.nunique()
        else:
            row.update({
                "voc_total_calls": 0, "voc_unique_contacts": 0, "voc_total_duration": 0,
                "voc_avg_duration": 0, "voc_max_duration": 0, "voc_std_duration": 0,
                "voc_active_days": 0, "voc_active_hours": 0
            })

        # --- SMS Features ---
        sms = df_u[df_u["source"] == "SMS"]
        if len(sms) > 0:
            row["sms_total_msgs"] = len(sms)
            row["sms_unique_contacts"] = sms["opposite_no_m"].nunique() if "opposite_no_m" in sms.columns else 0
            row["sms_active_hours"] = sms["event_time"].dt.hour.nunique()
            if "calltype_id" in sms.columns:
                calltype = pd.to_numeric(sms["calltype_id"], errors="coerce")
                row["sms_calltype_ratio"] = (calltype == 1).mean()
            else:
                row["sms_calltype_ratio"] = 0
        else:
            row.update({
                "sms_total_msgs": 0, "sms_unique_contacts": 0,
                "sms_active_hours": 0, "sms_calltype_ratio": 0
            })

        # --- App Features ---
        app = df_u[df_u["source"] == "APP"]
        if len(app) > 0:
            if "flow" in app.columns:
                flow = pd.to_numeric(app["flow"], errors="coerce").fillna(0)
            else:
                flow = pd.Series([0])

            row["app_months_active"] = app["month_id"].nunique() if "month_id" in app.columns else 0
            row["app_total_flow"] = flow.sum()
            row["app_avg_flow"] = flow.mean()
            row["app_std_flow"] = flow.std() if len(flow) > 1 else 0
            row["app_unique_apps_mean"] = app["busi_name"].nunique() if "busi_name" in app.columns else 0
            row["app_unique_apps_max"] = app["busi_name"].nunique() if "busi_name" in app.columns else 0
        else:
            row.update({
                "app_months_active": 0, "app_total_flow": 0, "app_avg_flow": 0,
                "app_std_flow": 0, "app_unique_apps_mean": 0, "app_unique_apps_max": 0
            })

        # --- User/ARPU Features ---
        arpu = df_u[df_u["source"] == "USER"]
        if len(arpu) > 0:
            if "arpu_value" in arpu.columns:
                arpu_val = pd.to_numeric(arpu["arpu_value"], errors="coerce")
            else:
                arpu_val = pd.Series([0])

            row["user_months_active"] = arpu["month_id"].nunique() if "month_id" in arpu.columns else 0
            row["arpu_mean"] = arpu_val.mean()
            row["arpu_std"] = arpu_val.std() if len(arpu_val) > 1 else 0
            row["arpu_max"] = arpu_val.max()
            row["idcard_cnt"] = arpu["idcard_cnt"].max() if "idcard_cnt" in arpu.columns else 0
        else:
            row.update({
                "user_months_active": 0, "arpu_mean": 0, "arpu_std": 0,
                "arpu_max": 0, "idcard_cnt": 0
            })

        # Meta features
        row["window_size"] = r
        row["snapshot_round"] = r
        row["label"] = label

        snapshot_rows.append(row)

    # Build DataFrame
    df_snapshot = pd.DataFrame(snapshot_rows).fillna(0)

    if df_snapshot.empty:
        return pd.DataFrame(), np.array([]), np.array([])

    y = df_snapshot["label"].values
    user_ids = df_snapshot["phone_no_m"].values
    X = df_snapshot.drop(columns=["phone_no_m", "label"])

    return X, y, user_ids



#  ▶ Classic Ml Snapshot based

###### Genrate input data

In [19]:

# ============================================================
# 📋 Define snapshot feature columns (same as sequencestreaming.py)
# ============================================================

SNAPSHOT_FEATURE_COLS = [
    # Voice
    "voc_total_calls", "voc_unique_contacts", "voc_total_duration",
    "voc_avg_duration", "voc_max_duration", "voc_std_duration",
    "voc_active_days", "voc_active_hours",
    # SMS
    "sms_total_msgs", "sms_unique_contacts", "sms_active_hours", "sms_calltype_ratio",
    # App
    "app_months_active", "app_total_flow", "app_avg_flow",
    "app_std_flow", "app_unique_apps_mean", "app_unique_apps_max",
    # User / ARPU
    "user_months_active", "arpu_mean", "arpu_std", "arpu_max", "idcard_cnt",
    # Meta
    "window_size", "snapshot_round"
]

print(f"📊 Snapshot features: {len(SNAPSHOT_FEATURE_COLS)} columns")

# ============================================================
# 🚀 RF/XGBoost - UNIFIED PIPELINE (Uses same events as LSTM)
# ============================================================


print("\n" + "="*60)
print(f"[{datetime.now()}] 🌲 RF/XGBoost Training (from events, same as LSTM)")
print("="*60)

# ============================================================
# 1️⃣ Generate training snapshots (r = max_seq_len)
# ============================================================
print(f"\n[{datetime.now()}] 📊 Generating training snapshots (r={max_seq_len})...")

X_train_snap, y_train_snap, users_train_snap = generate_snapshots_from_events(
    events_df=train_events_unscaled,
    users=train_users,
    r=max_seq_len,
    max_seq_len=max_seq_len,
    recent_mode=recent_mode
)
neg = (y_train_snap == 0).sum()
pos = (y_train_snap == 1).sum()
XGB_scale_pos_weight = neg / pos

print(f"[{datetime.now()}] ✅ Training snapshots: {len(X_train_snap)} users, {X_train_snap.shape[1]} features")

# ============================================================
# 2️⃣ Align columns and scale
# ============================================================
print(f"[{datetime.now()}] 🔄 Scaling...")
X_train_snap = X_train_snap.reindex(columns=SNAPSHOT_FEATURE_COLS, fill_value=0)
scaler_snap = StandardScaler().fit(X_train_snap)
X_train_scaled = scaler_snap.transform(X_train_snap)
print(f"[{datetime.now()}] ✅ Scaling done")


📊 Snapshot features: 25 columns

[2026-07-09 11:34:07.823903] 🌲 RF/XGBoost Training (from events, same as LSTM)

[2026-07-09 11:34:07.824026] 📊 Generating training snapshots (r=6)...


/tmp/ipykernel_993/10258993.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selected = events_filtered.groupby("phone_no_m", group_keys=False).apply(select_events)
Generating snapshots: 100%|██████████| 3907/3907 [00:11<00:00, 328.91it/s]


[2026-07-09 11:34:35.208025] ✅ Training snapshots: 3907 users, 25 features
[2026-07-09 11:34:35.208201] 🔄 Scaling...
[2026-07-09 11:34:35.214147] ✅ Scaling done


#### Show sample

In [20]:
# ============================================================
# 🔍 DEBUG: Print Sample Snapshots
# ============================================================

print("="*60)
print("🔍 SAMPLE SNAPSHOTS DEBUG")
print("="*60)

# Generate a small sample
X_sample, y_sample, users_sample = generate_snapshots_from_events(
    events_df=train_events_unscaled,
    users=list(train_users)[:10],  # Only 10 users for sample
    r=max_seq_len,
    max_seq_len=max_seq_len,
    recent_mode=recent_mode
)

print(f"\n📊 Sample shape: {X_sample.shape}")
print(f"📊 Labels: {y_sample}")
print(f"📊 Users: {users_sample}")

# Show features
print(f"\n📋 Feature columns ({len(X_sample.columns)}):")
print(X_sample.columns.tolist())

# Show sample data
print(f"\n📊 Sample snapshots (first 5 users):")
sample_df = X_sample.copy()
sample_df['label'] = y_sample
sample_df['user'] = users_sample
sample_df = sample_df[['user', 'label'] + [c for c in sample_df.columns if c not in ['user', 'label']]]
display(sample_df.head())

# Show statistics
print(f"\n📈 Feature statistics:")
display(X_sample.describe().T)

# Show class distribution
print(f"\n📊 Class distribution:")
print(f"   Fraud (1): {sum(y_sample == 1)} ({sum(y_sample == 1)/len(y_sample)*100:.1f}%)")
print(f"   Normal (0): {sum(y_sample == 0)} ({sum(y_sample == 0)/len(y_sample)*100:.1f}%)")

🔍 SAMPLE SNAPSHOTS DEBUG


/tmp/ipykernel_993/10258993.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selected = events_filtered.groupby("phone_no_m", group_keys=False).apply(select_events)
Generating snapshots: 100%|██████████| 10/10 [00:00<00:00, 315.38it/s]


📊 Sample shape: (10, 25)
📊 Labels: [0 0 1 0 0 0 1 0 0 0]
📊 Users: ['03ca40c4aa148f63dbb3beedf20c6424547d294741deb06d0a949c55c851b86e40b0e329da3cc98a3bf14100256eb283b94c0685155a3fa34b21022823536fab'
 '0a5344aa38fa437528731f1bfb3a847010984b29b7c82736b3f15d678ae57ba90fb8ca373c9243221256c43433373af9b1c6a2be4ff170f700b583e798feb420'
 '0c126c04959b5b79e8901ab6346508de8f7e1dd5f833b2967b4b97dd31a3aea4da6a6e3d38aed7f7b1fdafa718398d9f951bb8902ed68e79f48daf2f574a11fb'
 '207a8f04c4054de9db37e25a740ff6e7a8e912527ee22b39296420bf5bd9a1ccd5aa7423eb4c5790db88b2acba0bbb808fb1a85c2fdbe35dbc2a0a0096d95538'
 '4f73cf2699fe7d61103bd7a0977d91a64374a46ba17851b577e31dd8eff668fcf1f09dba497cbbbac730e36c20d1c47397c7ee0a3752d5768e4cfbaa061e5ed8'
 '4fd19a6cf38863dd07d6c42eeb6925739ede8c3ed726e3a9cdc4ae336bad220dadf1ea45f9940c812c2224b9d3b0da31fc982c5c6c5d7547891ee20975575362'
 '53d193a2d20f44e5c7467c66f7ba8dc97c570ebc3d0a91f88e593ba58810b3f38c41dc4735942a5d58c75e01cd9a5e3d3f90707ba89f7899955bc1b3407bcafc'
 'a561824

,user,label,voc_total_calls,voc_unique_contacts,voc_total_duration,voc_avg_duration,voc_max_duration,voc_std_duration,voc_active_days,voc_active_hours,sms_total_msgs,sms_unique_contacts,sms_active_hours,sms_calltype_ratio,app_months_active,app_total_flow,app_avg_flow,app_std_flow,app_unique_apps_mean,app_unique_apps_max,user_months_active,arpu_mean,arpu_std,arpu_max,idcard_cnt,window_size,snapshot_round
0,03ca40c4aa148f63dbb3beedf20c6424547d294741deb0...,0,5,5,152.0,30.4,68.0,21.524405,4,3,1,1,1,0.0,0,0.000000,0.000000,0.000000,0,0,0,0.0,0.0,0.0,0.0,6,6
1,0a5344aa38fa437528731f1bfb3a847010984b29b7c827...,0,2,2,146.0,73.0,114.0,57.982756,1,1,4,3,3,0.0,0,0.000000,0.000000,0.000000,0,0,0,0.0,0.0,0.0,0.0,6,6
2,0c126c04959b5b79e8901ab6346508de8f7e1dd5f833b2...,1,6,2,192.0,32.0,65.0,20.039960,1,1,0,0,0,0.0,0,0.000000,0.000000,0.000000,0,0,0,0.0,0.0,0.0,0.0,6,6
3,207a8f04c4054de9db37e25a740ff6e7a8e912527ee22b...,0,1,1,79.0,79.0,79.0,0.000000,1,1,0,0,0,0.0,1,0.004471,0.002235,0.001084,2,2,3,1.0,0.0,1.0,2.0,6,6
4,4f73cf2699fe7d61103bd7a0977d91a64374a46ba17851...,0,0,0,0.0,0.0,0.0,0.000000,0,0,6,2,2,0.0,0,0.000000,0.000000,0.000000,0,0,0,0.0,0.0,0.0,0.0,6,6



📈 Feature statistics:


,count,mean,std,min,25%,50%,75%,max
voc_total_calls,10.0,1.700000,2.162817,0.0,0.00,1.0,2.00000,6.000000
voc_unique_contacts,10.0,1.300000,1.567021,0.0,0.00,1.0,2.00000,5.000000
voc_total_duration,10.0,60.300000,75.836154,0.0,0.00,17.0,129.25000,192.000000
voc_avg_duration,10.0,24.140000,30.073547,0.0,0.00,13.5,31.60000,79.000000
voc_max_duration,10.0,35.300000,42.261225,0.0,0.00,13.5,67.25000,114.000000
voc_std_duration,10.0,9.954712,18.961766,0.0,0.00,0.0,15.02997,57.982756
voc_active_days,10.0,0.900000,1.197219,0.0,0.00,1.0,1.00000,4.000000
voc_active_hours,10.0,0.900000,0.994429,0.0,0.00,1.0,1.00000,3.000000
sms_total_msgs,10.0,3.800000,2.529822,0.0,1.75,4.5,6.00000,6.000000
sms_unique_contacts,10.0,1.400000,1.074968,0.0,1.00,1.0,2.00000,3.000000



📊 Class distribution:
   Fraud (1): 2 (20.0%)
   Normal (0): 8 (80.0%)


### RF and XGB NAS

In [21]:
# ============================================================
# 🔧 PREPARE DATA FOR RF/XGB NAS
# ============================================================
import optuna
from optuna.samplers import TPESampler
# Number of trials (same as TimesNet/LSTM NAS)
# ============================================================
# 📊 Trial Logging (matches NAS TimesNet structure)
# ============================================================
rf_trial_log = []
best_rf_f1_so_far = -1.0
best_rf_trial_so_far = -1
ENQUEUED_RF_IDS = {0, 1, 2, 3}

xgb_trial_log = []
best_xgb_f1_so_far = -1.0
best_xgb_trial_so_far = -1
ENQUEUED_XGB_IDS = {0, 1, 2, 3}
print(f"\\n[{datetime.now()}] 📊 Generating test snapshots...")

X_test_snap, y_test_snap, users_test_snap = generate_snapshots_from_events(
    events_df=test_events_unscaled,
    users=test_users,
    r=max_seq_len,
    max_seq_len=max_seq_len,
    recent_mode=recent_mode
)

X_test_snap = X_test_snap.reindex(columns=SNAPSHOT_FEATURE_COLS, fill_value=0)
X_test_scaled = scaler_snap.transform(X_test_snap)

print(f"[{datetime.now()}] ✅ Test snapshots: {len(X_test_snap)} users")

# Generate validation snapshots
print(f"\\n[{datetime.now()}] 📊 Generating validation snapshots...")

X_val_snap, y_val_snap, users_val_snap = generate_snapshots_from_events(
    events_df=val_events_unscaled,
    users=val_users,
    r=max_seq_len,
    max_seq_len=max_seq_len,
    recent_mode=recent_mode
)

X_val_snap = X_val_snap.reindex(columns=SNAPSHOT_FEATURE_COLS, fill_value=0)
X_val_scaled = scaler_snap.transform(X_val_snap)

print(f"[{datetime.now()}] ✅ Validation snapshots: {len(X_val_snap)} users")

neg = (y_train_snap == 0).sum()
pos = (y_train_snap == 1).sum()
XGB_scale_pos_weight = neg / pos
print(f"   Class ratio → Normal: {neg}, Fraud: {pos}, scale_pos_weight: {XGB_scale_pos_weight:.2f}")

# Already computed above for XGB, reuse the same neg/pos
RF_class_weight = {
    0: 1.0,           # normal users — keep as 1
    1: neg / pos      # fraud users — same ratio as XGB
}
print(f"   RF class_weight: {RF_class_weight}")


# ============================================================
# 🌲 RANDOM FOREST NAS
# ============================================================

def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
    }

    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    model.fit(X_train_scaled, y_train_snap)

    val_probs  = model.predict_proba(X_val_scaled)[:, 1]
    test_probs = model.predict_proba(X_test_scaled)[:, 1]

    # --- Validation threshold + metrics ---
    best_val_thr, best_val_f1 = find_best_threshold(y_val_snap, val_probs)
    val_preds = (val_probs >= best_val_thr).astype(int)
    val_auc = roc_auc_score(y_val_snap, val_probs)
    val_recall = recall_score(y_val_snap, val_preds, zero_division=0)
    val_precision = precision_score(y_val_snap, val_preds, zero_division=0)

    # --- Test metrics using VALIDATION threshold ---
    test_preds = (test_probs >= best_val_thr).astype(int)
    test_f1 = f1_score(y_test_snap, test_preds, zero_division=0)
    test_auc = roc_auc_score(y_test_snap, test_probs)
    test_recall = recall_score(y_test_snap, test_preds, zero_division=0)
    test_precision = precision_score(y_test_snap, test_preds, zero_division=0)

    # --- Oracle test threshold (monitoring only) ---
    best_test_thr, best_test_f1_oracle = find_best_threshold(y_test_snap, test_probs)

    # --- Log trial ---
    global best_rf_f1_so_far, best_rf_trial_so_far
    if best_val_f1 > best_rf_f1_so_far:
        best_rf_f1_so_far = best_val_f1
        best_rf_trial_so_far = trial.number

    trial_row = {
        "date": datetime.now().strftime("%Y-%m-%d"),
        "time": datetime.now().strftime("%H:%M:%S"),
        "seq_length": max_seq_len,
        "trial_id": trial.number,
        "model": "RandomForest",

        # Validation
        "val_f1": round(best_val_f1, 4),
        "val_auc": round(val_auc, 4),
        "val_recall": round(val_recall, 4),
        "val_precision": round(val_precision, 4),

        # Parameters
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "min_samples_split": params["min_samples_split"],
        "min_samples_leaf": params["min_samples_leaf"],

        # Thresholds
        "val_threshold": round(best_val_thr, 3),
        "best_test_threshold": round(best_test_thr, 3),

        # Best tracking
        "best_val_so_far": round(best_rf_f1_so_far, 4),
        "best_trial_id": best_rf_trial_so_far,

        # Test (monitoring only)
        "test_f1": round(test_f1, 4),
        "test_auc": round(test_auc, 4),
        "test_recall": round(test_recall, 4),
        "test_precision": round(test_precision, 4),

        # Diagnostics
        "gap_val_test": round(best_val_f1 - test_f1, 4),
        "is_enqueued": trial.number in ENQUEUED_RF_IDS,
        "status": "OK",
    }
    rf_trial_log.append(trial_row)

    display(pd.DataFrame([trial_row]))

    return best_val_f1

    # Evaluate on TEST set (same as TimesNet NAS)
    #probs = model.predict_proba(X_test_scaled)[:, 1]

    # # Same threshold search as TimesNet/LSTM NAS
    # best_f1 = -1.0
    # for thr in np.linspace(0.2, 0.8, 61):
    #     pred = (probs >= thr).astype(int)
    #     f1 = f1_score(y_test_snap, pred, zero_division=0)
    #     if f1 > best_f1:
    #         best_f1 = f1

    # return best_f1


# ============================================================
# 🚀 XGBOOST NAS
# ============================================================

def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 20),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "patience": trial.suggest_int("patience", 2, 5),
    }

    patience = params.pop("patience")
    model = XGBClassifier(**params,early_stopping_rounds=patience, random_state=42, n_jobs=-1, eval_metric='logloss')

    model.fit(X_train_scaled, y_train_snap,eval_set=[(X_val_scaled, y_val_snap)], verbose=False)
    actual_trees = model.best_iteration if hasattr(model, "best_iteration") else params["n_estimators"]

    val_probs  = model.predict_proba(X_val_scaled)[:, 1]
    test_probs = model.predict_proba(X_test_scaled)[:, 1]

    # --- Validation threshold + metrics ---
    best_val_thr, best_val_f1 = find_best_threshold(y_val_snap, val_probs)
    val_preds = (val_probs >= best_val_thr).astype(int)
    val_auc = roc_auc_score(y_val_snap, val_probs)
    val_recall = recall_score(y_val_snap, val_preds, zero_division=0)
    val_precision = precision_score(y_val_snap, val_preds, zero_division=0)

    # --- Test metrics using VALIDATION threshold ---
    test_preds = (test_probs >= best_val_thr).astype(int)
    test_f1 = f1_score(y_test_snap, test_preds, zero_division=0)
    test_auc = roc_auc_score(y_test_snap, test_probs)
    test_recall = recall_score(y_test_snap, test_preds, zero_division=0)
    test_precision = precision_score(y_test_snap, test_preds, zero_division=0)

    # --- Oracle test threshold (monitoring only) ---
    best_test_thr, best_test_f1_oracle = find_best_threshold(y_test_snap, test_probs)

    # --- Log trial ---
    global best_xgb_f1_so_far, best_xgb_trial_so_far
    if best_val_f1 > best_xgb_f1_so_far:
        best_xgb_f1_so_far = best_val_f1
        best_xgb_trial_so_far = trial.number

    trial_row = {
        "date": datetime.now().strftime("%Y-%m-%d"),
        "time": datetime.now().strftime("%H:%M:%S"),
        "seq_length": max_seq_len,
        "trial_id": trial.number,
        "model": "XGBoost",

        # Validation
        "val_f1": round(best_val_f1, 4),
        "val_auc": round(val_auc, 4),
        "val_recall": round(val_recall, 4),
        "val_precision": round(val_precision, 4),

        # Parameters
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "learning_rate": params["learning_rate"],
        "subsample": params["subsample"],
        "colsample_bytree": params["colsample_bytree"],
        "min_child_weight": params["min_child_weight"],
        "gamma": params["gamma"],

        "actual_trees": actual_trees,
        "patience": patience,

        # Thresholds
        "val_threshold": round(best_val_thr, 3),
        "best_test_threshold": round(best_test_thr, 3),

        # Best tracking
        "best_val_so_far": round(best_xgb_f1_so_far, 4),
        "best_trial_id": best_xgb_trial_so_far,

        # Test (monitoring only)
        "test_f1": round(test_f1, 4),
        "test_auc": round(test_auc, 4),
        "test_recall": round(test_recall, 4),
        "test_precision": round(test_precision, 4),

        # Diagnostics
        "gap_val_test": round(best_val_f1 - test_f1, 4),
        "is_enqueued": trial.number in ENQUEUED_XGB_IDS,
        "status": "OK",
    }
    xgb_trial_log.append(trial_row)

    display(pd.DataFrame([trial_row]))

    return best_val_f1


# ============================================================
# 🔬 RUN NAS - RANDOM FOREST
# ============================================================

print("\\n" + "="*60)
print("🌲 RANDOM FOREST NAS")
print("="*60)

sampler_rf = TPESampler(
    n_startup_trials=10,
    n_ei_candidates=24,
    multivariate=True,
    seed=42
)

study_rf = optuna.create_study(
    direction="maximize",
    sampler=sampler_rf
    #pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)

study_rf.enqueue_trial({"n_estimators": 100, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1})
study_rf.enqueue_trial({"n_estimators": 500, "max_depth": 20, "min_samples_split": 2, "min_samples_leaf": 1})
study_rf.enqueue_trial({"n_estimators": 200, "max_depth":  5, "min_samples_split":10, "min_samples_leaf": 5})
study_rf.enqueue_trial({"n_estimators": 300, "max_depth": 12, "min_samples_split": 5, "min_samples_leaf": 2})


print(f"[{datetime.now()}] 🚀 Starting RF NAS ({n_trials_rf_xgb} trials)...")
study_rf.optimize(rf_objective, n_trials=n_trials_rf_xgb)

print("\\n" + "="*60)
print("🎉 RF NAS COMPLETE")
print("="*60)
print(f"Best Trial: {study_rf.best_trial.number}")
print(f"Best F1: {study_rf.best_value:.4f}")
print(f"Best Params: {study_rf.best_params}")


# ============================================================
# 🔬 RUN NAS - XGBOOST
# ============================================================

print("\\n" + "="*60)
print("🚀 XGBOOST NAS")
print("="*60)

sampler_xgb = TPESampler(
    n_startup_trials=10,
    n_ei_candidates=24,
    multivariate=True,
    seed=42
)

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=sampler_xgb
    #pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)

study_xgb.enqueue_trial({"n_estimators": 100, "max_depth": 6, "learning_rate": 0.1,  "subsample": 1.0, "colsample_bytree": 1.0, "min_child_weight": 1, "gamma": 0.0, "patience": 3 })
study_xgb.enqueue_trial({"n_estimators": 500, "max_depth": 4, "learning_rate": 0.01, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 3, "gamma": 0.1, "patience": 3 })
study_xgb.enqueue_trial({"n_estimators":  50, "max_depth": 8, "learning_rate": 0.3,  "subsample": 0.7, "colsample_bytree": 0.7, "min_child_weight": 1, "gamma": 0.0, "patience": 3 })
study_xgb.enqueue_trial({"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.6, "colsample_bytree": 0.6, "min_child_weight": 5, "gamma": 0.5, "patience": 3 })



print(f"[{datetime.now()}] 🚀 Starting XGB NAS ({n_trials_rf_xgb} trials)...")
study_xgb.optimize(xgb_objective, n_trials=n_trials_rf_xgb)

print("\\n" + "="*60)
print("🎉 XGB NAS COMPLETE")
print("="*60)
print(f"Best Trial: {study_xgb.best_trial.number}")
print(f"Best F1: {study_xgb.best_value:.4f}")
print(f"Best Params: {study_xgb.best_params}")


# ============================================================
# 📊 SAVE RESULTS
# ============================================================

# Save Optuna study results
OUT_DIR = os.path.join(config["output"]["results_dir"], "NAS_v2/")
os.makedirs(OUT_DIR, exist_ok=True)
study_rf.trials_dataframe().to_csv(f"{OUT_DIR}nas_rf_results_WL{max_seq_len}.csv", index=False)
study_xgb.trials_dataframe().to_csv(f"{OUT_DIR}nas_xgb_results_WL{max_seq_len}.csv", index=False)
pd.DataFrame(rf_trial_log).to_csv(f"{OUT_DIR}nas_rf_trial_log_WL{max_seq_len}.csv", index=False)
pd.DataFrame(xgb_trial_log).to_csv(f"{OUT_DIR}nas_xgb_trial_log_WL{max_seq_len}.csv", index=False)

print("💾 Saved: nas_rf_results.csv, nas_xgb_results.csv")
print("💾 Saved: nas_rf_trial_log.csv, nas_xgb_trial_log.csv")

# ============================================================
# 📋 COMPARISON SUMMARY
# ============================================================

print("\n" + "="*60)
print("📋 NAS RESULTS COMPARISON")
print("="*60)

print(f"\n{'Model':<15} {'Best Val F1':<12} {'Best Trial':<12}")
print("-" * 40)
print(f"{'RandomForest':<15} {study_rf.best_value:<12.4f} {study_rf.best_trial.number:<12}")
print(f"{'XGBoost':<15} {study_xgb.best_value:<12.4f} {study_xgb.best_trial.number:<12}")

print("\n📊 Best RF Params:")
for k, v in study_rf.best_params.items():
    print(f"   {k}: {v}")

print("\n📊 Best XGB Params:")
for k, v in study_xgb.best_params.items():
    print(f"   {k}: {v}")

# Display full trial logs
print("\n📊 RF Trial Log (sorted by val_f1):")
display(
    pd.DataFrame(rf_trial_log)
    .sort_values("val_f1", ascending=False)
    .reset_index(drop=True)
)

print("\n📊 XGB Trial Log (sorted by val_f1):")
display(
    pd.DataFrame(xgb_trial_log)
    .sort_values("val_f1", ascending=False)
    .reset_index(drop=True)
)

\n[2026-07-09 11:34:36.366601] 📊 Generating test snapshots...


/tmp/ipykernel_993/10258993.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selected = events_filtered.groupby("phone_no_m", group_keys=False).apply(select_events)
Generating snapshots: 100%|██████████| 1222/1222 [00:03<00:00, 345.48it/s]


[2026-07-09 11:34:44.551165] ✅ Test snapshots: 1222 users
\n[2026-07-09 11:34:44.551291] 📊 Generating validation snapshots...


/tmp/ipykernel_993/10258993.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selected = events_filtered.groupby("phone_no_m", group_keys=False).apply(select_events)
Generating snapshots: 100%|██████████| 977/977 [00:02<00:00, 342.19it/s]
/tmp/ipykernel_993/3487339742.py:269: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler_rf = TPESampler(
[I 2026-07-09 11:34:51,002] A new study created in memory with name: no-name-0b22a1d1-82ed-46ab-b7ea-50ef8d1da630


[2026-07-09 11:34:50.999054] ✅ Validation snapshots: 977 users
   Class ratio → Normal: 2652, Fraud: 1255, scale_pos_weight: 2.11
   RF class_weight: {0: 1.0, 1: np.float64(2.113147410358566)}
\n============================================================
🌲 RANDOM FOREST NAS
[2026-07-09 11:34:51.003646] 🚀 Starting RF NAS (100 trials)...


...

[I 2026-07-09 11:37:14,642] Trial 99 finished with value: 0.6582984658298466 and parameters: {'n_estimators': 155, 'max_depth': 2, 'learning_rate': 0.018128948915756803, 'subsample': 0.9373595181841669, 'colsample_bytree': 0.9462636041256121, 'min_child_weight': 6, 'gamma': 0.8522937840444114, 'patience': 3}. Best is trial 47 with value: 0.6711590296495957.


\n============================================================
🎉 XGB NAS COMPLETE
Best Trial: 47
Best F1: 0.6712
Best Params: {'n_estimators': 136, 'max_depth': 5, 'learning_rate': 0.25702417209507145, 'subsample': 0.7380397704122255, 'colsample_bytree': 0.8336145544002741, 'min_child_weight': 9, 'gamma': 0.9886264497813226, 'patience': 2}
💾 Saved: nas_rf_results.csv, nas_xgb_results.csv
💾 Saved: nas_rf_trial_log.csv, nas_xgb_trial_log.csv

📋 NAS RESULTS COMPARISON

Model           Best Val F1  Best Trial  
----------------------------------------
RandomForest    0.6620       70          
XGBoost         0.6712       47          

📊 Best RF Params:
   n_estimators: 69
   max_depth: 7
   min_samples_split: 4
   min_samples_leaf: 5

📊 Best XGB Params:
   n_estimators: 136
   max_depth: 5
   learning_rate: 0.25702417209507145
   subsample: 0.7380397704122255
   colsample_bytree: 0.8336145544002741
   min_child_weight: 9
   gamma: 0.9886264497813226
   patience: 2

📊 RF Trial Log (sorted b

,date,time,seq_length,trial_id,model,val_f1,val_auc,val_recall,val_precision,n_estimators,max_depth,min_samples_split,min_samples_leaf,val_threshold,best_test_threshold,best_val_so_far,best_trial_id,test_f1,test_auc,test_recall,test_precision,gap_val_test,is_enqueued,status
0,2026-07-09,11:35:57,6,70,RandomForest,0.6620,0.8002,0.7611,0.5858,69,7,4,5,0.26,0.26,0.6620,70,0.6286,0.7803,0.6997,0.5705,0.0335,False,OK
1,2026-07-09,11:35:35,6,41,RandomForest,0.6620,0.7951,0.7516,0.5915,70,11,7,10,0.31,0.28,0.6620,41,0.6184,0.7743,0.6743,0.5711,0.0436,False,OK
2,2026-07-09,11:35:04,6,12,RandomForest,0.6611,0.7958,0.7516,0.5900,439,17,7,9,0.30,0.30,0.6611,6,0.6209,0.7724,0.6794,0.5717,0.0401,False,OK
3,2026-07-09,11:35:10,6,18,RandomForest,0.6611,0.7958,0.7516,0.5900,427,11,4,10,0.30,0.28,0.6611,6,0.6184,0.7745,0.6743,0.5711,0.0426,False,OK
4,2026-07-09,11:35:23,6,29,RandomForest,0.6611,0.7949,0.7516,0.5900,422,16,5,10,0.30,0.28,0.6611,6,0.6193,0.7704,0.6768,0.5708,0.0417,False,OK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026-07-09,11:35:49,6,61,RandomForest,0.6432,0.7876,0.7261,0.5772,230,2,17,2,0.43,0.43,0.6620,41,0.6019,0.7686,0.6539,0.5575,0.0413,False,OK
96,2026-07-09,11:35:53,6,66,RandomForest,0.6432,0.7874,0.7261,0.5772,279,2,13,10,0.43,0.43,0.6620,41,0.6019,0.7684,0.6539,0.5575,0.0413,False,OK
97,2026-07-09,11:36:00,6,75,RandomForest,0.6397,0.7857,0.6306,0.6492,51,3,5,3,0.39,0.20,0.6620,70,0.5792,0.7680,0.5445,0.6185,0.0606,False,OK
98,2026-07-09,11:34:53,6,1,RandomForest,0.6386,0.7781,0.7006,0.5867,500,20,2,1,0.32,0.32,0.6537,0,0.6043,0.7558,0.6412,0.5714,0.0343,True,OK



📊 XGB Trial Log (sorted by val_f1):


,date,time,seq_length,trial_id,model,val_f1,val_auc,val_recall,val_precision,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,gamma,actual_trees,patience,val_threshold,best_test_threshold,best_val_so_far,best_trial_id,test_f1,test_auc,test_recall,test_precision,gap_val_test,is_enqueued,status
0,2026-07-09,11:36:49,6,47,XGBoost,0.6712,0.8054,0.7930,0.5818,136,5,0.257024,0.738040,0.833615,9,0.988626,8,2,0.25,0.28,0.6712,47,0.6256,0.7812,0.7226,0.5515,0.0456,False,OK
1,2026-07-09,11:37:13,6,97,XGBoost,0.6703,0.8061,0.7803,0.5875,450,5,0.017285,0.940933,0.958300,5,0.489365,151,2,0.26,0.27,0.6712,47,0.6247,0.7814,0.7074,0.5594,0.0456,False,OK
2,2026-07-09,11:36:38,6,20,XGBoost,0.6694,0.8054,0.7834,0.5843,149,4,0.113332,0.941360,0.819457,6,0.325454,25,2,0.26,0.26,0.6694,18,0.6290,0.7801,0.7074,0.5662,0.0404,False,OK
3,2026-07-09,11:36:39,6,24,XGBoost,0.6694,0.8075,0.7803,0.5861,102,4,0.060039,0.961119,0.863240,8,0.848727,52,3,0.26,0.31,0.6694,18,0.6258,0.7837,0.7150,0.5564,0.0436,False,OK
4,2026-07-09,11:36:37,6,18,XGBoost,0.6694,0.8021,0.7771,0.5880,295,2,0.111008,0.948012,0.673945,6,0.064221,43,3,0.29,0.29,0.6694,18,0.6279,0.7819,0.6997,0.5694,0.0416,False,OK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026-07-09,11:36:33,6,8,XGBoost,0.6429,0.7933,0.7739,0.5498,79,20,0.246597,0.904199,0.652307,1,0.684233,8,3,0.29,0.42,0.6657,1,0.6022,0.7673,0.7048,0.5256,0.0407,False,OK
96,2026-07-09,11:36:53,6,59,XGBoost,0.6355,0.7863,0.6720,0.6029,154,15,0.001021,0.613957,0.570838,10,0.476191,153,4,0.33,0.32,0.6712,47,0.5860,0.7716,0.5980,0.5746,0.0495,False,OK
97,2026-07-09,11:36:45,6,34,XGBoost,0.6300,0.7790,0.6752,0.5905,82,2,0.004809,0.913266,0.955340,10,0.991465,81,3,0.33,0.28,0.6694,18,0.5862,0.7633,0.6056,0.5680,0.0438,False,OK
98,2026-07-09,11:36:54,6,60,XGBoost,0.6271,0.7581,0.6051,0.6507,93,2,0.002840,0.959622,0.824055,5,0.939009,92,2,0.33,0.29,0.6712,47,0.5734,0.7338,0.5267,0.6292,0.0537,False,OK


#freeze

In [22]:
%pip freeze > "{project_path}requirement/freez/NASEnhancedPretraindMLModleAdvance-lock.txt"

In [23]:
end = time.time()
elapsed = end - start_time

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)

print(f"Total Time = {hours}h {minutes}m {seconds}s")

Total Time = 0h 6m 36s
